
读取冻结的最终Master中Center A的180例，加载锁定NPZ与120个Stage-B checkpoints进行GPU外部验证推理。该流程不重新训练、选择checkpoint、调整阈值或校准概率。

In [ ]:
# -*- coding: utf-8 -*-
r"""
NPC重度急性口腔黏膜炎3D dose-map项目
================================================================================

核心原则
--------
1. V2为最终Development版本，不再训练、不再改label、不再重划分。
2. Development = Center B + Center C, n=309。
3. External = Center A, n=180；label 0/1 = 88/92。
4. 外部验证只加载V2训练完成的Stage-B checkpoints，不执行任何训练。
5. 每个CNN模型共有4 repeats × 5 folds = 20个Stage-B checkpoints。
6. 对Center A：
   - 保存每个checkpoint的180例预测；
   - 每个repeat内5-fold模型等权平均 -> 4个repeat-ensemble；
   - 20个checkpoint全部等权平均 -> 正式external ensemble prediction。
7. 外部集绝不用于：
   - 模型选择；
   - checkpoint选择；
   - threshold调优；
   - calibration重新拟合；
   - label修改。
8. threshold：
   - 0.5仅作描述；
   - 正式threshold指标另使用Development patient-averaged OOF锁定的Youden threshold。
9. 主要评价：
   AUC、PR-AUC、Brier、bootstrap 95% CI、校准、Sensitivity/Specificity等。
10. 配对bootstrap：
   - M2-Abl vs M2
   - M2-Abl vs M3-Abl（GTV空间信息增量）
   - M3-Abl vs M3
   - M3 vs M4
"""

# 必须放在import torch之前
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

from pathlib import Path
from datetime import datetime
import gc
import hashlib
import json
import sys
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    auc,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve

warnings.filterwarnings("ignore", category=RuntimeWarning)


# =============================================================================
# 1. 锁定路径与版本
# =============================================================================

def require_env_path(name: str) -> Path:
    """Return a required absolute path from an environment variable."""
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"Set {name} to an absolute path before running this notebook."
        )
    path = Path(value).expanduser()
    if not path.is_absolute():
        raise RuntimeError(f"{name} must be an absolute path: {value!r}")
    return path.resolve()

PROJECT_DIR = require_env_path("NPC_PROJECT_ROOT")

MASTER_XLSX = (
    PROJECT_DIR
    / "NPC_3DCNN_mucositis_master_frozen_BCdev_Aexternal_v2.xlsx"
)

NPZ_DIR = (
    PROJECT_DIR
    / "preprocessed_497_2x2x3_patch80x112x64_v2"
    / "npz"
)

SPLIT_ROOT = (
    PROJECT_DIR
    / "fixed_splits_repeated_5fold_4repeats_497_BCdev_Aexternal_v2"
)

SPLITS_LOCKED_JSON = (
    SPLIT_ROOT
    / "SPLITS_LOCKED.json"
)

TRAIN_ROOT = (
    PROJECT_DIR
    / "formal_main_models_no_scalar_4x5_497_BCdev_Aexternal_v2"
)

AGGREGATE_DIR = (
    TRAIN_ROOT
    / "aggregate"
)

TRAIN_COMPLETED_JSON = (
    TRAIN_ROOT
    / "FORMAL_4X5_COMPLETED.json"
)

TRAIN_AGGREGATE_AUDIT_JSON = (
    AGGREGATE_DIR
    / "formal_4x5_aggregate_audit.json"
)

EXTERNAL_OUTPUT_ROOT = (
    PROJECT_DIR
    / "external_validation_CenterA_FINAL180_RERUN_v2"
)

EXTERNAL_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_SEED = 20260812

BATCH_SIZE = 4
NUM_WORKERS = 0

EXPECTED_SHAPE_ZYX = (
    64,
    112,
    80,
)

BASE_CHANNELS = 16
DROPOUT = 0.20
DESCRIPTIVE_THRESHOLD = 0.5

REPEATS = [1, 2, 3, 4]
FOLDS = [1, 2, 3, 4, 5]

MODEL_CHANNELS = {
    "M1_Dose_Only": [
        "dose",
    ],
    "M2_Abl": [
        "dose",
        "oral",
    ],
    "M2_Oral_MaskedDose": [
        "dose",
        "oral",
        "dose_oral",
    ],
    "M3_Abl": [
        "dose",
        "oral",
        "gtv",
    ],
    "M3_Oral_GTV_MaskedDose": [
        "dose",
        "oral",
        "gtv",
        "dose_oral",
        "dose_gtv",
    ],
    "M4_CT_Augmented_M3": [
        "ct",
        "dose",
        "oral",
        "gtv",
        "dose_oral",
        "dose_gtv",
    ],
}

MODEL_DISPLAY_NAMES = {
    "M1_Dose_Only": "M1",
    "M2_Abl": "M2-Abl",
    "M2_Oral_MaskedDose": "M2",
    "M3_Abl": "M3-Abl",
    "M3_Oral_GTV_MaskedDose": "M3",
    "M4_CT_Augmented_M3": "M4",
}

MODELS = list(
    MODEL_CHANNELS.keys()
)

DEV_OOF_FILES = {
    "M1_Dose_Only":
        "M1_Dose_Only_patient_averaged_oof_309.csv",
    "M2_Abl":
        "M2_Abl_patient_averaged_oof_309.csv",
    "M2_Oral_MaskedDose":
        "M2_Oral_MaskedDose_patient_averaged_oof_309.csv",
    "M3_Abl":
        "M3_Abl_patient_averaged_oof_309.csv",
    "M3_Oral_GTV_MaskedDose":
        "M3_Oral_GTV_MaskedDose_patient_averaged_oof_309.csv",
    "M4_CT_Augmented_M3":
        "M4_CT_Augmented_M3_patient_averaged_oof_309.csv",
}

PAIRWISE_COMPARISONS = [
    (
        "M2-Abl vs M2",
        "M2_Abl",
        "M2_Oral_MaskedDose",
    ),
    (
        "M2-Abl vs M3-Abl",
        "M2_Abl",
        "M3_Abl",
    ),
    (
        "M3-Abl vs M3",
        "M3_Abl",
        "M3_Oral_GTV_MaskedDose",
    ),
    (
        "M3 vs M4",
        "M3_Oral_GTV_MaskedDose",
        "M4_CT_Augmented_M3",
    ),
]

ALLOWED_IMAGE_CHANNELS = {
    "ct",
    "dose",
    "oral",
    "gtv",
    "dose_oral",
    "dose_gtv",
}


# =============================================================================
# 2. 通用函数
# =============================================================================

def sha256_file(
    file_path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            block = file.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def save_json(
    payload,
    path: Path,
) -> None:
    def convert(value):
        if isinstance(value, Path):
            return str(value)

        if isinstance(value, dict):
            return {
                str(key): convert(item)
                for key, item
                in value.items()
            }

        if isinstance(value, (list, tuple)):
            return [
                convert(item)
                for item in value
            ]

        if isinstance(value, np.generic):
            return value.item()

        return value

    path.write_text(
        json.dumps(
            convert(payload),
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )


def safe_auc(
    y_true,
    probabilities,
):
    try:
        return float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        )
    except Exception:
        return np.nan


def safe_pr_auc(
    y_true,
    probabilities,
):
    """Trapezoidal PR-AUC with recall on the x-axis (manuscript definition)."""
    try:
        precision, recall, _ = precision_recall_curve(
            y_true,
            probabilities,
        )
        return float(auc(recall, precision))
    except Exception:
        return np.nan


def safe_brier(
    y_true,
    probabilities,
):
    try:
        return float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        )
    except Exception:
        return np.nan


def calculate_threshold_metrics(
    y_true,
    probabilities,
    threshold,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    y_pred = (
        probabilities
        >= float(threshold)
    ).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = [
        int(x)
        for x in cm.ravel()
    ]

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "sensitivity": float(
            recall_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "specificity": float(
            specificity
        ),
        "precision": float(
            precision_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def calculate_probability_metrics(
    y_true,
    probabilities,
):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    return {
        "roc_auc": safe_auc(
            y_true,
            probabilities,
        ),
        "pr_auc": safe_pr_auc(
            y_true,
            probabilities,
        ),
        "brier": safe_brier(
            y_true,
            probabilities,
        ),
        "probability_mean": float(
            np.mean(probabilities)
        ),
        "probability_std": float(
            np.std(
                probabilities,
                ddof=1,
            )
        ),
        "probability_min": float(
            np.min(probabilities)
        ),
        "probability_max": float(
            np.max(probabilities)
        ),
    }


def development_youden_threshold(
    y_true,
    probabilities,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    fpr, tpr, thresholds = roc_curve(
        y_true,
        probabilities,
    )

    finite = np.isfinite(
        thresholds
    )

    fpr = fpr[finite]
    tpr = tpr[finite]
    thresholds = thresholds[finite]

    youden = (
        tpr - fpr
    )

    best_value = np.max(
        youden
    )

    candidate_indices = np.where(
        np.isclose(
            youden,
            best_value,
            atol=1e-12,
            rtol=0,
        )
    )[0]

    if len(candidate_indices) > 1:
        chosen_index = candidate_indices[
            np.argmin(
                np.abs(
                    thresholds[
                        candidate_indices
                    ]
                    - 0.5
                )
            )
        ]
    else:
        chosen_index = int(
            candidate_indices[0]
        )

    return {
        "threshold": float(
            thresholds[
                chosen_index
            ]
        ),
        "development_sensitivity": float(
            tpr[
                chosen_index
            ]
        ),
        "development_specificity": float(
            1.0
            - fpr[
                chosen_index
            ]
        ),
        "development_youden": float(
            youden[
                chosen_index
            ]
        ),
    }


def stratified_bootstrap_metric_ci(
    y_true,
    probabilities,
    metric_name,
    iterations=2000,
    seed=20260812,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    rng = np.random.default_rng(
        seed
    )

    idx0 = np.where(
        y_true == 0
    )[0]

    idx1 = np.where(
        y_true == 1
    )[0]

    if (
        len(idx0) == 0
        or
        len(idx1) == 0
    ):
        return {
            "estimate": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "n_boot_valid": 0,
        }

    if metric_name == "roc_auc":
        metric_fn = safe_auc
    elif metric_name == "pr_auc":
        metric_fn = safe_pr_auc
    elif metric_name == "brier":
        metric_fn = safe_brier
    else:
        raise ValueError(
            f"未知metric_name: {metric_name}"
        )

    estimate = metric_fn(
        y_true,
        probabilities,
    )

    values = []

    for _ in range(
        int(iterations)
    ):
        sample0 = rng.choice(
            idx0,
            size=len(idx0),
            replace=True,
        )

        sample1 = rng.choice(
            idx1,
            size=len(idx1),
            replace=True,
        )

        sample = np.concatenate(
            [
                sample0,
                sample1,
            ]
        )

        rng.shuffle(
            sample
        )

        value = metric_fn(
            y_true[
                sample
            ],
            probabilities[
                sample
            ],
        )

        if np.isfinite(
            value
        ):
            values.append(
                value
            )

    values = np.asarray(
        values,
        dtype=float,
    )

    return {
        "estimate": float(
            estimate
        ),
        "ci_low": float(
            np.quantile(
                values,
                0.025,
            )
        ),
        "ci_high": float(
            np.quantile(
                values,
                0.975,
            )
        ),
        "n_boot_valid": int(
            len(values)
        ),
    }


def paired_stratified_bootstrap_delta(
    y_true,
    reference_probabilities,
    candidate_probabilities,
    metric_name,
    iterations=2000,
    seed=20260812,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    reference_probabilities = np.asarray(
        reference_probabilities,
        dtype=float,
    )

    candidate_probabilities = np.asarray(
        candidate_probabilities,
        dtype=float,
    )

    if metric_name == "roc_auc":
        metric_fn = safe_auc
        higher_is_better = True
    elif metric_name == "pr_auc":
        metric_fn = safe_pr_auc
        higher_is_better = True
    elif metric_name == "brier":
        metric_fn = safe_brier
        higher_is_better = False
    else:
        raise ValueError(
            f"未知metric_name: {metric_name}"
        )

    reference_estimate = metric_fn(
        y_true,
        reference_probabilities,
    )

    candidate_estimate = metric_fn(
        y_true,
        candidate_probabilities,
    )

    raw_delta = (
        candidate_estimate
        - reference_estimate
    )

    if higher_is_better:
        improvement = (
            candidate_estimate
            - reference_estimate
        )
    else:
        improvement = (
            reference_estimate
            - candidate_estimate
        )

    idx0 = np.where(
        y_true == 0
    )[0]

    idx1 = np.where(
        y_true == 1
    )[0]

    rng = np.random.default_rng(
        seed
    )

    boot_improvements = []

    for _ in range(
        int(iterations)
    ):
        sample = np.concatenate(
            [
                rng.choice(
                    idx0,
                    size=len(idx0),
                    replace=True,
                ),
                rng.choice(
                    idx1,
                    size=len(idx1),
                    replace=True,
                ),
            ]
        )

        rng.shuffle(
            sample
        )

        ref_value = metric_fn(
            y_true[
                sample
            ],
            reference_probabilities[
                sample
            ],
        )

        cand_value = metric_fn(
            y_true[
                sample
            ],
            candidate_probabilities[
                sample
            ],
        )

        if (
            np.isfinite(ref_value)
            and
            np.isfinite(cand_value)
        ):
            if higher_is_better:
                boot_improvements.append(
                    cand_value
                    - ref_value
                )
            else:
                boot_improvements.append(
                    ref_value
                    - cand_value
                )

    boot_improvements = np.asarray(
        boot_improvements,
        dtype=float,
    )

    p_lower = np.mean(
        boot_improvements <= 0
    )

    p_upper = np.mean(
        boot_improvements >= 0
    )

    p_two_sided = min(
        1.0,
        2.0
        * min(
            p_lower,
            p_upper,
        ),
    )

    return {
        "reference_estimate": float(
            reference_estimate
        ),
        "candidate_estimate": float(
            candidate_estimate
        ),
        "raw_candidate_minus_reference": float(
            raw_delta
        ),
        "improvement_positive_direction": float(
            improvement
        ),
        "improvement_ci_low": float(
            np.quantile(
                boot_improvements,
                0.025,
            )
        ),
        "improvement_ci_high": float(
            np.quantile(
                boot_improvements,
                0.975,
            )
        ),
        "p_two_sided": float(
            p_two_sided
        ),
        "n_boot_valid": int(
            len(
                boot_improvements
            )
        ),
    }


def calibration_intercept_slope(
    y_true,
    probabilities,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    p = np.clip(
        np.asarray(
            probabilities,
            dtype=float,
        ),
        1e-6,
        1.0 - 1e-6,
    )

    logits = np.log(
        p / (1.0 - p)
    ).reshape(-1, 1)

    try:
        model = LogisticRegression(
            penalty=None,
            solver="lbfgs",
            max_iter=5000,
        )
        model.fit(
            logits,
            y_true,
        )
    except Exception:
        model = LogisticRegression(
            C=1e6,
            solver="lbfgs",
            max_iter=5000,
        )
        model.fit(
            logits,
            y_true,
        )

    return {
        "calibration_intercept": float(
            model.intercept_[0]
        ),
        "calibration_slope": float(
            model.coef_[0][0]
        ),
    }


# =============================================================================
# 3. Master审计 + 建立External dataframe
# =============================================================================

if not MASTER_XLSX.exists():
    raise FileNotFoundError(
        f"V2 Master不存在：{MASTER_XLSX}"
    )

current_master_sha256 = sha256_file(
    MASTER_XLSX
)

master = pd.read_excel(
    MASTER_XLSX,
    sheet_name=0,
    dtype=object,
    engine="openpyxl",
)

master.columns = [
    str(column).strip()
    for column
    in master.columns
]

required_master_columns = {
    "patient_id",
    "model_center",
    "exclude_reason",
    "severe_mucositis",
}

missing = (
    required_master_columns
    - set(
        master.columns
    )
)

if missing:
    raise KeyError(
        f"Master缺少字段：{sorted(missing)}"
    )

master = master.loc[
    master[
        "patient_id"
    ].notna()
].copy()

master["patient_id"] = pd.to_numeric(
    master["patient_id"],
    errors="raise",
).astype(int)

if master[
    "patient_id"
].duplicated().any():
    raise RuntimeError(
        "Master存在重复patient_id"
    )

if len(master) != 540:
    raise RuntimeError(
        f"Master病例行数错误：{len(master)} != 540"
    )

exclude_text = (
    master[
        "exclude_reason"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)

included = master.loc[
    exclude_text.eq("")
].copy()

excluded = master.loc[
    ~exclude_text.eq("")
].copy()

if len(included) != 489:
    raise RuntimeError(
        f"正式纳入病例错误：{len(included)} != 489"
    )

if len(excluded) != 51:
    raise RuntimeError(
        f"排除病例错误：{len(excluded)} != 51"
    )

included[
    "model_center"
] = (
    included[
        "model_center"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)

included[
    "severe_mucositis"
] = pd.to_numeric(
    included[
        "severe_mucositis"
    ],
    errors="raise",
).astype(int)

if not set(
    included[
        "severe_mucositis"
    ].unique()
).issubset({0, 1}):
    raise RuntimeError(
        "Master存在非0/1标签"
    )

center_counts = (
    included[
        "model_center"
    ]
    .value_counts()
    .to_dict()
)

if center_counts != {
    "A": 180,
    "B": 81,
    "C": 228,
}:
    raise RuntimeError(
        f"中心数量错误：{center_counts}"
    )

center_label_counts = {
    (
        str(center),
        int(label),
    ): int(count)
    for (
        center,
        label,
    ), count
    in included.groupby(
        [
            "model_center",
            "severe_mucositis",
        ]
    ).size().items()
}

if center_label_counts != {
    ("A", 0): 88,
    ("A", 1): 92,
    ("B", 0): 42,
    ("B", 1): 39,
    ("C", 0): 116,
    ("C", 1): 112,
}:
    raise RuntimeError(
        "中心×标签数量错误："
        f"{center_label_counts}"
    )

development_df = (
    included.loc[
        included[
            "model_center"
        ].isin(
            [
                "B",
                "C",
            ]
        )
    ]
    .copy()
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)

external_df = (
    included.loc[
        included[
            "model_center"
        ].eq(
            "A"
        )
    ]
    .copy()
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)

if len(development_df) != 309:
    raise RuntimeError(
        "Development病例数不是309"
    )

if len(external_df) != 180:
    raise RuntimeError(
        "External病例数不是180"
    )

if (
    external_df[
        "severe_mucositis"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
    != {
        0: 88,
        1: 92,
    }
):
    raise RuntimeError(
        "External标签分布不是88/92"
    )


# =============================================================================
# 4. 扫描497个NPZ，以NPZ内部patient_id建立路径映射
#    只读取patient_id用于ID核对；不读取NPZ label/cohort。
# =============================================================================

if not NPZ_DIR.exists():
    raise FileNotFoundError(
        f"NPZ目录不存在：{NPZ_DIR}"
    )

npz_files = sorted(
    NPZ_DIR.glob(
        "*.npz"
    )
)

if len(npz_files) != 497:
    raise RuntimeError(
        f"NPZ文件数错误：{len(npz_files)} != 497"
    )

npz_lookup = {}

print(
    "扫描497个NPZ内部patient_id；"
    "不读取NPZ label/cohort……"
)

for index, npz_path in enumerate(
    npz_files,
    start=1,
):
    with np.load(
        npz_path,
        allow_pickle=False,
    ) as data:
        if (
            "patient_id"
            not in data.files
        ):
            raise RuntimeError(
                f"NPZ缺少patient_id：{npz_path}"
            )

        patient_id = int(
            np.asarray(
                data[
                    "patient_id"
                ]
            ).reshape(-1)[0]
        )

    if patient_id in npz_lookup:
        raise RuntimeError(
            f"NPZ出现重复patient_id：{patient_id}"
        )

    npz_lookup[
        patient_id
    ] = npz_path

    if (
        index % 50 == 0
        or
        index == len(npz_files)
    ):
        print(
            f"NPZ ID扫描：{index}/497"
        )

included_ids = set(
    included[
        "patient_id"
    ].astype(int)
)

missing_npz_ids = sorted(
    included_ids
    - set(
        npz_lookup.keys()
    )
)

extra_npz_ids = sorted(
    set(
        npz_lookup.keys()
    )
    - included_ids
)

# 只要求最终Master纳入的489例全部有NPZ。
# NPZ目录允许保留已被最终Master排除的历史病例文件。
if missing_npz_ids:
    raise RuntimeError(
        "最终Master纳入病例存在缺失NPZ。\n"
        f"缺少：{missing_npz_ids[:20]}"
    )

if extra_npz_ids:
    print(
        "提示：NPZ目录中保留了已被最终Master排除的历史病例，"
        "本次分析不会使用它们：",
        extra_npz_ids,
    )

external_df[
    "npz_path"
] = (
    external_df[
        "patient_id"
    ]
    .map(
        npz_lookup
    )
)

if external_df[
    "npz_path"
].isna().any():
    raise RuntimeError(
        "External存在找不到NPZ的病例"
    )

external_df[
    "cohort"
] = "External"

external_df[
    "stratum"
] = (
    external_df[
        "model_center"
    ]
    + "_label"
    + external_df[
        "severe_mucositis"
    ].astype(str)
)

external_df[
    [
        "patient_id",
        "model_center",
        "cohort",
        "severe_mucositis",
        "stratum",
        "npz_path",
    ]
].to_csv(
    EXTERNAL_OUTPUT_ROOT
    / "CenterA_external_FINAL180_cases.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# =============================================================================

if not TRAIN_ROOT.exists():
    raise FileNotFoundError(
        f"V2正式训练目录不存在：{TRAIN_ROOT}"
    )

checkpoint_rows = []

for repeat_number in REPEATS:
    for fold_number in FOLDS:
        for model_name in MODELS:
            checkpoint_path = (
                TRAIN_ROOT
                / f"repeat_{repeat_number:02d}"
                / f"fold_{fold_number:02d}"
                / model_name
                / "stage_B_outer_train_final_model.pt"
            )

            if not checkpoint_path.exists():
                raise FileNotFoundError(
                    "缺少Stage-B checkpoint：\n"
                    f"{checkpoint_path}"
                )

            checkpoint_rows.append({
                "repeat": int(
                    repeat_number
                ),
                "fold": int(
                    fold_number
                ),
                "model_name": model_name,
                "display_name": (
                    MODEL_DISPLAY_NAMES[
                        model_name
                    ]
                ),
                "checkpoint_path": str(
                    checkpoint_path
                ),
            })

checkpoint_manifest = pd.DataFrame(
    checkpoint_rows
)

if len(
    checkpoint_manifest
) != 120:
    raise RuntimeError(
        "Stage-B checkpoint总数不是120"
    )

checkpoint_manifest.to_csv(
    EXTERNAL_OUTPUT_ROOT
    / "V2_stageB_120_checkpoints_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 6. Development-only阈值锁定
#    此步骤只读取309例Development OOF；不读取External预测结果。
# =============================================================================

locked_threshold_rows = []

for model_name in MODELS:
    oof_path = (
        AGGREGATE_DIR
        / DEV_OOF_FILES[
            model_name
        ]
    )

    if not oof_path.exists():
        raise FileNotFoundError(
            "缺少V2 Development patient-averaged OOF：\n"
            f"{oof_path}"
        )

    dev_oof = pd.read_csv(
        oof_path,
        encoding="utf-8-sig",
    )

    probability_candidates = [
        column
        for column
        in dev_oof.columns
        if str(column).lower()
        in {
            "mean_probability",
            "probability_mean",
            "avg_probability",
            "average_probability",
            "probability",
        }
    ]

    if not probability_candidates:
        probability_candidates = [
            column
            for column
            in dev_oof.columns
            if (
                "prob"
                in str(column).lower()
                and
                pd.api.types.is_numeric_dtype(
                    dev_oof[
                        column
                    ]
                )
            )
        ]

    if not probability_candidates:
        raise RuntimeError(
            f"{model_name} OOF找不到概率列："
            f"{list(dev_oof.columns)}"
        )

    probability_column = (
        probability_candidates[0]
    )

    label_candidates = [
        column
        for column
        in dev_oof.columns
        if str(column).lower()
        in {
            "y_true",
            "severe_mucositis",
            "label",
            "true_label",
        }
    ]

    if not label_candidates:
        raise RuntimeError(
            f"{model_name} OOF找不到label列"
        )

    label_column = (
        label_candidates[0]
    )

    if len(dev_oof) != 309:
        raise RuntimeError(
            f"{model_name} patient OOF不是309例"
        )

    threshold_info = (
        development_youden_threshold(
            y_true=dev_oof[
                label_column
            ].astype(int).to_numpy(),
            probabilities=dev_oof[
                probability_column
            ].astype(float).to_numpy(),
        )
    )

    locked_threshold_rows.append({
        "model_name": model_name,
        "display_name": (
            MODEL_DISPLAY_NAMES[
                model_name
            ]
        ),
        "development_oof_file": str(
            oof_path
        ),
        "probability_column": (
            probability_column
        ),
        "label_column": (
            label_column
        ),
        "threshold_rule": (
            "Youden index from locked V2 "
            "Development patient-averaged OOF"
        ),
        **threshold_info,
    })

locked_thresholds = pd.DataFrame(
    locked_threshold_rows
)

locked_thresholds.to_csv(
    EXTERNAL_OUTPUT_ROOT
    / "development_locked_youden_thresholds.csv",
    index=False,
    encoding="utf-8-sig",
)

locked_threshold_lookup = dict(
    zip(
        locked_thresholds[
            "model_name"
        ],
        locked_thresholds[
            "threshold"
        ],
    )
)

print()
print(
    "Development-only thresholds已锁定并保存。"
)
print(
    locked_thresholds[
        [
            "display_name",
            "threshold",
            "development_sensitivity",
            "development_specificity",
        ]
    ].to_string(
        index=False
    )
)


# =============================================================================
# =============================================================================

class NPCNPZDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        channels,
    ):
        self.dataframe = (
            dataframe
            .copy()
            .reset_index(
                drop=True
            )
        )

        self.channels = list(
            channels
        )

        unknown_channels = (
            set(
                self.channels
            )
            - ALLOWED_IMAGE_CHANNELS
        )

        if unknown_channels:
            raise ValueError(
                "未知图像通道："
                f"{sorted(unknown_channels)}"
            )

    def __len__(
        self,
    ):
        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.dataframe.iloc[
            index
        ]

        patient_id = int(
            row[
                "patient_id"
            ]
        )

        label = int(
            row[
                "severe_mucositis"
            ]
        )

        npz_path = Path(
            row[
                "npz_path"
            ]
        )

        required_base_channels = {
            channel
            for channel
            in self.channels
            if channel
            in {
                "ct",
                "dose",
                "oral",
                "gtv",
            }
        }

        if (
            "dose_oral"
            in self.channels
        ):
            required_base_channels.update(
                {
                    "dose",
                    "oral",
                }
            )

        if (
            "dose_gtv"
            in self.channels
        ):
            required_base_channels.update(
                {
                    "dose",
                    "gtv",
                }
            )

        arrays = {}

        with np.load(
            npz_path,
            allow_pickle=False,
        ) as data:
            for channel in sorted(
                required_base_channels
            ):
                if channel not in data:
                    raise KeyError(
                        f"病例{patient_id}"
                        f"缺少基础通道{channel}"
                    )

                array = np.asarray(
                    data[
                        channel
                    ],
                    dtype=np.float32,
                )

                if (
                    tuple(
                        array.shape
                    )
                    != EXPECTED_SHAPE_ZYX
                ):
                    raise RuntimeError(
                        f"病例{patient_id}"
                        f"通道{channel}shape错误："
                        f"{array.shape}"
                    )

                if not np.isfinite(
                    array
                ).all():
                    raise RuntimeError(
                        f"病例{patient_id}"
                        f"通道{channel}存在NaN/Inf"
                    )

                arrays[
                    channel
                ] = array

            internal_patient_id = int(
                np.asarray(
                    data[
                        "patient_id"
                    ]
                ).reshape(-1)[0]
            )

        if (
            internal_patient_id
            != patient_id
        ):
            raise RuntimeError(
                f"病例{patient_id} NPZ内部ID错误："
                f"{internal_patient_id}"
            )

        if "oral" in arrays:
            oral = (
                arrays[
                    "oral"
                ]
                > 0.5
            ).astype(
                np.float32
            )

            if int(
                oral.sum()
            ) <= 0:
                raise RuntimeError(
                    f"病例{patient_id} oral为空"
                )

            arrays[
                "oral"
            ] = oral

        if "gtv" in arrays:
            gtv = (
                arrays[
                    "gtv"
                ]
                > 0.5
            ).astype(
                np.float32
            )

            if int(
                gtv.sum()
            ) <= 0:
                raise RuntimeError(
                    f"病例{patient_id} gtv为空"
                )

            arrays[
                "gtv"
            ] = gtv

        if (
            "dose_oral"
            in self.channels
        ):
            arrays[
                "dose_oral"
            ] = (
                arrays[
                    "dose"
                ]
                * arrays[
                    "oral"
                ]
            ).astype(
                np.float32,
                copy=False,
            )

        if (
            "dose_gtv"
            in self.channels
        ):
            arrays[
                "dose_gtv"
            ] = (
                arrays[
                    "dose"
                ]
                * arrays[
                    "gtv"
                ]
            ).astype(
                np.float32,
                copy=False,
            )

        image = np.stack(
            [
                arrays[
                    channel
                ]
                for channel
                in self.channels
            ],
            axis=0,
        ).astype(
            np.float32,
            copy=False,
        )

        expected_shape = (
            len(
                self.channels
            ),
            *EXPECTED_SHAPE_ZYX,
        )

        if tuple(
            image.shape
        ) != expected_shape:
            raise RuntimeError(
                f"病例{patient_id}最终输入shape错误："
                f"{image.shape} != {expected_shape}"
            )

        return (
            torch.from_numpy(
                np.ascontiguousarray(
                    image
                )
            ),
            torch.tensor(
                float(label),
                dtype=torch.float32,
            ),
            torch.tensor(
                patient_id,
                dtype=torch.int64,
            ),
        )


# =============================================================================
# =============================================================================

def make_group_norm(
    channels: int,
) -> nn.GroupNorm:
    groups = 8

    while (
        groups > 1
        and
        channels % groups != 0
    ):
        groups //= 2

    return nn.GroupNorm(
        num_groups=groups,
        num_channels=channels,
    )


class BasicResidualBlock3D(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
    ):
        super().__init__()

        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )

        self.norm1 = make_group_norm(
            out_channels
        )

        self.relu = nn.ReLU(
            inplace=True
        )

        self.conv2 = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )

        self.norm2 = make_group_norm(
            out_channels
        )

        if (
            stride != 1
            or
            in_channels != out_channels
        ):
            self.shortcut = nn.Sequential(
                nn.Conv3d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                make_group_norm(
                    out_channels
                ),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(
        self,
        x,
    ):
        identity = self.shortcut(
            x
        )

        x = self.conv1(
            x
        )
        x = self.norm1(
            x
        )
        x = self.relu(
            x
        )

        x = self.conv2(
            x
        )
        x = self.norm2(
            x
        )

        x = x + identity
        x = self.relu(
            x
        )

        return x


class LightweightResNet10_3D(nn.Module):
    def __init__(
        self,
        in_channels: int,
        base_channels: int = 16,
        dropout: float = 0.20,
    ):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv3d(
                in_channels,
                base_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            make_group_norm(
                base_channels
            ),
            nn.ReLU(
                inplace=True
            ),
            nn.MaxPool3d(
                kernel_size=2,
                stride=2,
            ),
        )

        self.layer1 = BasicResidualBlock3D(
            base_channels,
            base_channels,
            stride=1,
        )
        self.layer2 = BasicResidualBlock3D(
            base_channels,
            base_channels * 2,
            stride=2,
        )
        self.layer3 = BasicResidualBlock3D(
            base_channels * 2,
            base_channels * 4,
            stride=2,
        )
        self.layer4 = BasicResidualBlock3D(
            base_channels * 4,
            base_channels * 8,
            stride=2,
        )

        self.global_pool = nn.AdaptiveAvgPool3d(
            output_size=1
        )

        self.dropout = nn.Dropout(
            p=dropout
        )

        self.classifier = nn.Linear(
            base_channels * 8,
            1,
        )

        self._initialize_weights()

    def _initialize_weights(
        self,
    ):
        for module in self.modules():
            if isinstance(
                module,
                nn.Conv3d,
            ):
                nn.init.kaiming_normal_(
                    module.weight,
                    mode="fan_out",
                    nonlinearity="relu",
                )
            elif isinstance(
                module,
                nn.Linear,
            ):
                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=0.01,
                )

                if (
                    module.bias
                    is not None
                ):
                    nn.init.zeros_(
                        module.bias
                    )

    def forward(
        self,
        x,
    ):
        x = self.stem(
            x
        )
        x = self.layer1(
            x
        )
        x = self.layer2(
            x
        )
        x = self.layer3(
            x
        )
        x = self.layer4(
            x
        )
        x = self.global_pool(
            x
        )
        x = torch.flatten(
            x,
            1,
        )
        x = self.dropout(
            x
        )

        return self.classifier(
            x
        ).squeeze(1)


# =============================================================================
# 9. 推理辅助函数
# =============================================================================

def make_loader(
    dataframe,
    channels,
):
    dataset = NPCNPZDataset(
        dataframe=dataframe,
        channels=channels,
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
    )


def amp_context_from_checkpoint(
    checkpoint,
    device,
):
    amp_mode = str(
        checkpoint.get(
            "amp_mode",
            "FP32",
        )
    ).upper()

    if (
        device.type
        != "cuda"
        or
        amp_mode == "FP32"
    ):
        return {
            "enabled": False,
            "dtype": torch.float32,
            "mode": "FP32",
        }

    if amp_mode == "BF16":
        return {
            "enabled": True,
            "dtype": torch.bfloat16,
            "mode": "BF16",
        }

    if amp_mode == "FP16":
        return {
            "enabled": True,
            "dtype": torch.float16,
            "mode": "FP16",
        }

    return {
        "enabled": False,
        "dtype": torch.float32,
        "mode": "FP32",
    }


@torch.inference_mode()
def predict_checkpoint(
    checkpoint_path,
    expected_model_name,
    expected_channels,
    dataframe,
    device,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    checkpoint_model_name = str(
        checkpoint.get(
            "model_name",
            ""
        )
    )

    if (
        checkpoint_model_name
        != expected_model_name
    ):
        raise RuntimeError(
            f"checkpoint模型名错误："
            f"{checkpoint_model_name} != "
            f"{expected_model_name}\n"
            f"{checkpoint_path}"
        )

    checkpoint_channels = list(
        checkpoint.get(
            "channels",
            []
        )
    )

    if (
        checkpoint_channels
        != list(
            expected_channels
        )
    ):
        raise RuntimeError(
            f"checkpoint channels错误："
            f"{checkpoint_channels} != "
            f"{expected_channels}\n"
            f"{checkpoint_path}"
        )

    if (
        checkpoint.get(
            "external_validation_used",
            False
        )
        is not False
    ):
        raise RuntimeError(
            "发现训练checkpoint声明"
            "external_validation_used不是False："
            f"{checkpoint_path}"
        )

    model = LightweightResNet10_3D(
        in_channels=len(
            expected_channels
        ),
        base_channels=BASE_CHANNELS,
        dropout=DROPOUT,
    ).to(
        device
    )

    model.load_state_dict(
        checkpoint[
            "model_state"
        ],
        strict=True,
    )

    model.eval()

    loader = make_loader(
        dataframe=dataframe,
        channels=expected_channels,
    )

    amp_info = amp_context_from_checkpoint(
        checkpoint,
        device,
    )

    all_patient_ids = []
    all_labels = []
    all_probabilities = []

    for (
        images,
        labels,
        patient_ids,
    ) in loader:
        images = images.to(
            device,
            non_blocking=True,
        )

        if amp_info[
            "enabled"
        ]:
            with torch.autocast(
                device_type="cuda",
                dtype=amp_info[
                    "dtype"
                ],
            ):
                logits = model(
                    images
                )
        else:
            logits = model(
                images
            )

        probabilities = (
            torch.sigmoid(
                logits.float()
            )
            .detach()
            .cpu()
            .numpy()
            .astype(float)
        )

        all_patient_ids.extend(
            patient_ids
            .cpu()
            .numpy()
            .astype(int)
            .tolist()
        )

        all_labels.extend(
            labels
            .cpu()
            .numpy()
            .astype(int)
            .tolist()
        )

        all_probabilities.extend(
            probabilities
            .tolist()
        )

    result = pd.DataFrame({
        "patient_id": all_patient_ids,
        "y_true": all_labels,
        "probability": all_probabilities,
    })

    if len(result) != 180:
        raise RuntimeError(
            f"checkpoint预测病例数错误：{len(result)} != 180"
        )

    if result[
        "patient_id"
    ].duplicated().any():
        raise RuntimeError(
            "checkpoint预测存在重复patient_id"
        )

    result = (
        result
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )

    expected_ids = (
        dataframe[
            "patient_id"
        ]
        .astype(int)
        .sort_values()
        .to_numpy()
    )

    if not np.array_equal(
        result[
            "patient_id"
        ].to_numpy(),
        expected_ids,
    ):
        raise RuntimeError(
            "checkpoint external patient_id集合错误"
        )

    del model
    del loader
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return (
        result,
        amp_info[
            "mode"
        ],
        int(
            checkpoint.get(
                "selected_epoch",
                -1,
            )
        ),
    )


# =============================================================================
# 10. 正式Center A推理：6模型 × 20 checkpoints
# =============================================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA不可用；正式外部验证停止。"
    )

device = torch.device(
    "cuda"
)

print()
print("=" * 110)
print(
)
print("=" * 110)
print(
    "Python：",
    sys.version.split()[0],
)
print(
    "PyTorch：",
    torch.__version__,
)
print(
    "GPU：",
    torch.cuda.get_device_name(0),
)
print(
    "Master：",
    MASTER_XLSX.name,
)
print(
    "Master SHA256：",
    current_master_sha256,
)
print(
    "Development：309例（锁定，不重新训练）"
)
print(
    "External：Center A = 180例，0/1=88/92"
)
print(
    "正式external predictor："
    "每模型20个Stage-B checkpoints等权ensemble"
)
print("=" * 110)

all_long_predictions = []

for model_index, model_name in enumerate(
    MODELS,
    start=1,
):
    channels = (
        MODEL_CHANNELS[
            model_name
        ]
    )

    print()
    print(
        f"[{model_index}/6] "
        f"{MODEL_DISPLAY_NAMES[model_name]} "
        f"{channels}"
    )

    for repeat_number in REPEATS:
        for fold_number in FOLDS:
            checkpoint_path = (
                TRAIN_ROOT
                / f"repeat_{repeat_number:02d}"
                / f"fold_{fold_number:02d}"
                / model_name
                / "stage_B_outer_train_final_model.pt"
            )

            prediction_df, amp_mode, selected_epoch = (
                predict_checkpoint(
                    checkpoint_path=checkpoint_path,
                    expected_model_name=model_name,
                    expected_channels=channels,
                    dataframe=external_df,
                    device=device,
                )
            )

            prediction_df[
                "model_name"
            ] = model_name

            prediction_df[
                "display_name"
            ] = (
                MODEL_DISPLAY_NAMES[
                    model_name
                ]
            )

            prediction_df[
                "repeat"
            ] = int(
                repeat_number
            )

            prediction_df[
                "fold"
            ] = int(
                fold_number
            )

            prediction_df[
                "amp_mode"
            ] = amp_mode

            prediction_df[
                "selected_epoch"
            ] = int(
                selected_epoch
            )

            all_long_predictions.append(
                prediction_df
            )

            print(
                f"  repeat_{repeat_number:02d}/"
                f"fold_{fold_number:02d} PASS"
            )

external_long = pd.concat(
    all_long_predictions,
    ignore_index=True,
)

expected_long_rows = (
    180
    * 20
    * 6
)

if len(
    external_long
) != expected_long_rows:
    raise RuntimeError(
        f"External long行数错误："
        f"{len(external_long)} != "
        f"{expected_long_rows}"
    )

external_long.to_csv(
    EXTERNAL_OUTPUT_ROOT
    / "external_all_models_all_20checkpoints_long.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# =============================================================================

repeat_ensemble = (
    external_long
    .groupby(
        [
            "model_name",
            "display_name",
            "repeat",
            "patient_id",
            "y_true",
        ],
        as_index=False,
    )
    .agg(
        probability=(
            "probability",
            "mean",
        ),
        fold_probability_sd=(
            "probability",
            "std",
        ),
        n_fold_models=(
            "fold",
            "nunique",
        ),
    )
)

if not (
    repeat_ensemble[
        "n_fold_models"
    ]
    .eq(5)
    .all()
):
    raise RuntimeError(
        "某些repeat-ensemble不是5个fold模型"
    )

if len(
    repeat_ensemble
) != (
    180
    * 4
    * 6
):
    raise RuntimeError(
        "Repeat-ensemble行数错误"
    )

repeat_ensemble.to_csv(
    EXTERNAL_OUTPUT_ROOT
    / "external_repeat_ensemble_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

repeat_metric_rows = []

for (
    model_name,
    display_name,
    repeat_number,
), group in repeat_ensemble.groupby(
    [
        "model_name",
        "display_name",
        "repeat",
    ]
):
    group = (
        group
        .sort_values(
            "patient_id"
        )
    )

    probability_metrics = (
        calculate_probability_metrics(
            group[
                "y_true"
            ].to_numpy(),
            group[
                "probability"
            ].to_numpy(),
        )
    )

    threshold_metrics = (
        calculate_threshold_metrics(
            group[
                "y_true"
            ].to_numpy(),
            group[
                "probability"
            ].to_numpy(),
            threshold=(
                locked_threshold_lookup[
                    model_name
                ]
            ),
        )
    )

    repeat_metric_rows.append({
        "model_name": model_name,
        "display_name": display_name,
        "repeat": int(
            repeat_number
        ),
        **probability_metrics,
        **{
            f"locked_{key}": value
            for key, value
            in threshold_metrics.items()
        },
    })

repeat_metrics = pd.DataFrame(
    repeat_metric_rows
)

repeat_metrics.to_csv(
    EXTERNAL_OUTPUT_ROOT
    / "external_repeat_ensemble_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 12. 正式20-model ensemble：每例平均20个checkpoint概率
# =============================================================================

patient_ensemble = (
    external_long
    .groupby(
        [
            "model_name",
            "display_name",
            "patient_id",
            "y_true",
        ],
        as_index=False,
    )
    .agg(
        probability=(
            "probability",
            "mean",
        ),
        checkpoint_probability_sd=(
            "probability",
            "std",
        ),
        probability_min=(
            "probability",
            "min",
        ),
        probability_max=(
            "probability",
            "max",
        ),
        n_checkpoints=(
            "probability",
            "size",
        ),
    )
)

if not (
    patient_ensemble[
        "n_checkpoints"
    ]
    .eq(20)
    .all()
):
    raise RuntimeError(
        "某些患者不是20个checkpoint平均"
    )

if len(
    patient_ensemble
) != (
    180
    * 6
):
    raise RuntimeError(
        "正式patient-ensemble行数错误"
    )

patient_ensemble.to_csv(
    EXTERNAL_OUTPUT_ROOT
    / "external_patient_20model_ensemble_all_models.csv",
    index=False,
    encoding="utf-8-sig",
)

for model_name in MODELS:
    patient_ensemble.loc[
        patient_ensemble[
            "model_name"
        ].eq(
            model_name
        )
    ].to_csv(
        EXTERNAL_OUTPUT_ROOT
        / (
            f"{model_name}"
            "_CenterA_180_20model_ensemble_predictions.csv"
        ),
        index=False,
        encoding="utf-8-sig",
    )


# =============================================================================
# 13. 正式External metrics + bootstrap 95% CI + calibration
# =============================================================================

final_metric_rows = []
calibration_rows = []

for model_index, model_name in enumerate(
    MODELS,
    start=1,
):
    model_df = (
        patient_ensemble.loc[
            patient_ensemble[
                "model_name"
            ].eq(
                model_name
            )
        ]
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )

    y_true = (
        model_df[
            "y_true"
        ]
        .astype(int)
        .to_numpy()
    )

    probabilities = (
        model_df[
            "probability"
        ]
        .astype(float)
        .to_numpy()
    )

    base_metrics = (
        calculate_probability_metrics(
            y_true,
            probabilities,
        )
    )

    auc_ci = (
        stratified_bootstrap_metric_ci(
            y_true,
            probabilities,
            "roc_auc",
            iterations=BOOTSTRAP_ITERATIONS,
            seed=(
                BOOTSTRAP_SEED
                + model_index * 101
            ),
        )
    )

    pr_ci = (
        stratified_bootstrap_metric_ci(
            y_true,
            probabilities,
            "pr_auc",
            iterations=BOOTSTRAP_ITERATIONS,
            seed=(
                BOOTSTRAP_SEED
                + model_index * 103
            ),
        )
    )

    brier_ci = (
        stratified_bootstrap_metric_ci(
            y_true,
            probabilities,
            "brier",
            iterations=BOOTSTRAP_ITERATIONS,
            seed=(
                BOOTSTRAP_SEED
                + model_index * 107
            ),
        )
    )

    descriptive_metrics = (
        calculate_threshold_metrics(
            y_true,
            probabilities,
            threshold=DESCRIPTIVE_THRESHOLD,
        )
    )

    locked_threshold = float(
        locked_threshold_lookup[
            model_name
        ]
    )

    locked_metrics = (
        calculate_threshold_metrics(
            y_true,
            probabilities,
            threshold=locked_threshold,
        )
    )

    calibration_metrics = (
        calibration_intercept_slope(
            y_true,
            probabilities,
        )
    )

    repeat_subset = (
        repeat_metrics.loc[
            repeat_metrics[
                "model_name"
            ].eq(
                model_name
            )
        ]
    )

    final_metric_rows.append({
        "model_name": model_name,
        "display_name": (
            MODEL_DISPLAY_NAMES[
                model_name
            ]
        ),
        "n_external": 180,
        "n_negative": 88,
        "n_positive": 92,

        "roc_auc": (
            base_metrics[
                "roc_auc"
            ]
        ),
        "roc_auc_ci_low": (
            auc_ci[
                "ci_low"
            ]
        ),
        "roc_auc_ci_high": (
            auc_ci[
                "ci_high"
            ]
        ),

        "pr_auc": (
            base_metrics[
                "pr_auc"
            ]
        ),
        "pr_auc_ci_low": (
            pr_ci[
                "ci_low"
            ]
        ),
        "pr_auc_ci_high": (
            pr_ci[
                "ci_high"
            ]
        ),

        "brier": (
            base_metrics[
                "brier"
            ]
        ),
        "brier_ci_low": (
            brier_ci[
                "ci_low"
            ]
        ),
        "brier_ci_high": (
            brier_ci[
                "ci_high"
            ]
        ),

        "calibration_intercept": (
            calibration_metrics[
                "calibration_intercept"
            ]
        ),
        "calibration_slope": (
            calibration_metrics[
                "calibration_slope"
            ]
        ),

        "repeat_ensemble_auc_mean": float(
            repeat_subset[
                "roc_auc"
            ].mean()
        ),
        "repeat_ensemble_auc_sd": float(
            repeat_subset[
                "roc_auc"
            ].std(
                ddof=1
            )
        ),

        "checkpoint_probability_sd_mean": float(
            model_df[
                "checkpoint_probability_sd"
            ].mean()
        ),

        "threshold_0p5": 0.5,
        "accuracy_0p5": (
            descriptive_metrics[
                "accuracy"
            ]
        ),
        "balanced_accuracy_0p5": (
            descriptive_metrics[
                "balanced_accuracy"
            ]
        ),
        "sensitivity_0p5": (
            descriptive_metrics[
                "sensitivity"
            ]
        ),
        "specificity_0p5": (
            descriptive_metrics[
                "specificity"
            ]
        ),
        "precision_0p5": (
            descriptive_metrics[
                "precision"
            ]
        ),
        "f1_0p5": (
            descriptive_metrics[
                "f1"
            ]
        ),
        "tn_0p5": (
            descriptive_metrics[
                "tn"
            ]
        ),
        "fp_0p5": (
            descriptive_metrics[
                "fp"
            ]
        ),
        "fn_0p5": (
            descriptive_metrics[
                "fn"
            ]
        ),
        "tp_0p5": (
            descriptive_metrics[
                "tp"
            ]
        ),

        "development_locked_youden_threshold": (
            locked_threshold
        ),
        "accuracy_locked": (
            locked_metrics[
                "accuracy"
            ]
        ),
        "balanced_accuracy_locked": (
            locked_metrics[
                "balanced_accuracy"
            ]
        ),
        "sensitivity_locked": (
            locked_metrics[
                "sensitivity"
            ]
        ),
        "specificity_locked": (
            locked_metrics[
                "specificity"
            ]
        ),
        "precision_locked": (
            locked_metrics[
                "precision"
            ]
        ),
        "f1_locked": (
            locked_metrics[
                "f1"
            ]
        ),
        "tn_locked": (
            locked_metrics[
                "tn"
            ]
        ),
        "fp_locked": (
            locked_metrics[
                "fp"
            ]
        ),
        "fn_locked": (
            locked_metrics[
                "fn"
            ]
        ),
        "tp_locked": (
            locked_metrics[
                "tp"
            ]
        ),
    })

    fraction_true, mean_predicted = (
        calibration_curve(
            y_true,
            probabilities,
            n_bins=10,
            strategy="quantile",
        )
    )

    for bin_index, (
        observed,
        predicted,
    ) in enumerate(
        zip(
            fraction_true,
            mean_predicted,
        ),
        start=1,
    ):
        calibration_rows.append({
            "model_name": model_name,
            "display_name": (
                MODEL_DISPLAY_NAMES[
                    model_name
                ]
            ),
            "bin": int(
                bin_index
            ),
            "mean_predicted_probability": float(
                predicted
            ),
            "observed_event_rate": float(
                observed
            ),
        })

final_metrics = pd.DataFrame(
    final_metric_rows
)

final_metrics.to_csv(
    EXTERNAL_OUTPUT_ROOT
    / "external_final_20model_ensemble_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

calibration_df = pd.DataFrame(
    calibration_rows
)

calibration_df.to_csv(
    EXTERNAL_OUTPUT_ROOT
    / "external_calibration_10_quantile_bins.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 14. External配对bootstrap
# =============================================================================

wide = (
    patient_ensemble[
        [
            "patient_id",
            "y_true",
            "model_name",
            "probability",
        ]
    ]
    .pivot(
        index=[
            "patient_id",
            "y_true",
        ],
        columns="model_name",
        values="probability",
    )
    .reset_index()
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)

pair_rows = []

for pair_index, (
    comparison_name,
    reference_model,
    candidate_model,
) in enumerate(
    PAIRWISE_COMPARISONS,
    start=1,
):
    for metric_index, metric_name in enumerate(
        [
            "roc_auc",
            "pr_auc",
            "brier",
        ],
        start=1,
    ):
        result = (
            paired_stratified_bootstrap_delta(
                y_true=wide[
                    "y_true"
                ].to_numpy(),
                reference_probabilities=wide[
                    reference_model
                ].to_numpy(),
                candidate_probabilities=wide[
                    candidate_model
                ].to_numpy(),
                metric_name=metric_name,
                iterations=BOOTSTRAP_ITERATIONS,
                seed=(
                    BOOTSTRAP_SEED
                    + pair_index * 1000
                    + metric_index * 37
                ),
            )
        )

        pair_rows.append({
            "comparison": comparison_name,
            "reference_model": reference_model,
            "reference_display_name": (
                MODEL_DISPLAY_NAMES[
                    reference_model
                ]
            ),
            "candidate_model": candidate_model,
            "candidate_display_name": (
                MODEL_DISPLAY_NAMES[
                    candidate_model
                ]
            ),
            "metric": metric_name,
            **result,
        })

paired_results = pd.DataFrame(
    pair_rows
)

paired_results.to_csv(
    EXTERNAL_OUTPUT_ROOT
    / "external_paired_bootstrap_comparisons.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 15. 汇总审计与Excel
# =============================================================================

audit = {
    "status": "PASS",
    "created_at": datetime.now().isoformat(),
    "master_filename": MASTER_XLSX.name,
    "master_sha256": current_master_sha256,
    "development_status": "locked; not retrained",
    "development_definition": "model_center B+C",
    "development_n": 309,
    "external_definition": "model_center A",
    "external_n": 180,
    "external_label0": 88,
    "external_label1": 92,
    "npz_count": int(
        len(
            npz_files
        )
    ),
    "npz_label_read": False,
    "npz_cohort_read": False,
    "external_threshold_tuning": False,
    "threshold_source": (
        "locked V2 Development patient-averaged OOF"
    ),
    "checkpoint_source": (
    ),
    "n_checkpoints_total": int(
        len(
            checkpoint_manifest
        )
    ),
    "n_checkpoints_per_model": 20,
    "ensemble_weighting": "equal",
    "repeat_ensemble": (
        "mean of 5 fold checkpoints within each repeat"
    ),
    "final_external_ensemble": (
        "mean of all 20 Stage-B checkpoints per model"
    ),
    "external_prediction_rows_long": int(
        len(
            external_long
        )
    ),
    "external_repeat_ensemble_rows": int(
        len(
            repeat_ensemble
        )
    ),
    "external_patient_ensemble_rows": int(
        len(
            patient_ensemble
        )
    ),
    "bootstrap_iterations": int(
        BOOTSTRAP_ITERATIONS
    ),
    "external_used_for_model_selection": False,
    "external_used_for_checkpoint_selection": False,
    "external_used_for_retraining": False,
}

save_json(
    audit,
    EXTERNAL_OUTPUT_ROOT
    / "EXTERNAL_VALIDATION_AUDIT.json",
)

with pd.ExcelWriter(
    EXTERNAL_OUTPUT_ROOT
    / "external_validation_summary.xlsx",
    engine="openpyxl",
) as writer:
    final_metrics.to_excel(
        writer,
        sheet_name="Final_metrics",
        index=False,
    )

    repeat_metrics.to_excel(
        writer,
        sheet_name="Repeat_metrics",
        index=False,
    )

    locked_thresholds.to_excel(
        writer,
        sheet_name="Dev_locked_thresholds",
        index=False,
    )

    paired_results.to_excel(
        writer,
        sheet_name="Paired_bootstrap",
        index=False,
    )

    calibration_df.to_excel(
        writer,
        sheet_name="Calibration_bins",
        index=False,
    )

    checkpoint_manifest.to_excel(
        writer,
        sheet_name="Checkpoint_manifest",
        index=False,
    )

print()
print("=" * 110)
print(
    "CENTER A独立外部验证完成"
)
print("=" * 110)
print(
    final_metrics[
        [
            "display_name",
            "roc_auc",
            "roc_auc_ci_low",
            "roc_auc_ci_high",
            "pr_auc",
            "brier",
            "repeat_ensemble_auc_mean",
            "repeat_ensemble_auc_sd",
            "development_locked_youden_threshold",
            "sensitivity_locked",
            "specificity_locked",
        ]
    ].to_string(
        index=False,
    )
)
print()
print(
    "输出目录：",
    EXTERNAL_OUTPUT_ROOT,
)
print(
    "核心汇总：",
    EXTERNAL_OUTPUT_ROOT
    / "external_validation_summary.xlsx",
)
print(
    "审计：",
    EXTERNAL_OUTPUT_ROOT
    / "EXTERNAL_VALIDATION_AUDIT.json",
)
print("=" * 110)